# Экспорт данных из OSM


> 🚧 Раздел в разработке.


OpenStreetMap — это огромная открытая база геоданных, которую создают сами пользователи. Иногда её называют «Википедией карт», потому что любой желающий может добавить объекты.

Данные **OpenStreetMap (OSM)** распространяются по лицензии **Open Database License (ODbL 1.0)**. Это означает, что данные можно свободно скачивать, изменять, дополнять и даже использовать в коммерческих проектах, но с обязательным указанием источника:

© OpenStreetMap contributors

В OSM все добавляемые объекты строго структурированы: каждый из них описывается набором тегов, которые определяют его характеристики и назначение.

> В этом разделе мы познакомимся со структурой данных OSM, разберёмся, как работают теги, и научимся выгружать определённые типы объектов на заданную территорию с помощью Python.


## 0. Импорт библиотек


In [ ]:
import osmnx as ox

- [**OSMnx**](https://osmnx.readthedocs.io/) (`osmnx`) — библиотека Python для загрузки, анализа и визуализации данных **OpenStreetMap**.
  Она позволяет легко получать пространственные объекты по названию места или по координатам.


## 1. Геокодирование границ

Перед тем как выгружать данные из OpenStreetMap, необходимо определить территорию исследования. Один из самых простых способов — задать её по названию места.

В OSMnx для этого используется функция `ox.geocode_to_gdf()`, которая геокодирует название и возвращает границы в формате GeoDataFrame.


In [ ]:
area_name = "Центральный район, Санкт-Петербург"

admin_district = ox.geocode_to_gdf(area_name)

Посмотрим на данные на карте c помощью метода `explore()`:


In [ ]:
admin_district.explore(tiles='cartodbpositron')

Мы получили корректные административные границы района. Это означает, что в дальнейшем мы можем использовать то же название места для загрузки конкретных объектов или уличной сети


## 2. Выгрузка объектов по тегам

В OpenStreetMap объекты описываются с помощью тегов — пар «ключ–значение», которые определяют их тип и характеристики.

Именно теги позволяют нам формировать выборку данных. Мы можем указать, какие объекты нас интересуют, и выгрузить их для ранее заданной территории.

Список тегов OSM доступен здесь: [Map Features](https://wiki.openstreetmap.org/wiki/Map_features).

**Как задаются теги в OSMnx**

В OSMnx теги передаются в виде словаря Python, где:

- ключ словаря — это ключ тега в OSM
- значение — конкретное значение тега или `True`, если нас интересуют все объекты с этим ключом

Да, можно прямо в Markdown аккуратно встроить примеры — без лишних пояснений, но наглядно. Вот вариант оформления:

Например:

**Все здания:**

```python
tags = {"building": True}
```

**Только кафе:**

```python
tags = {"amenity": "cafe"}
```

**Кафе, рестораны и бары:**

```python
tags = {"amenity": ["cafe", "restaurant", "bar"]}
```


### 2.1 По названию места


Самый простой способ выгрузки данных — запрос по названию территории. Если границы объекта корректно определяются в OpenStreetMap (а это мы проверили для нашего района выше), можно сразу использовать это название для получения нужных объектов.

Такой способ удобен, когда мы работаем с административными единицами (районами, городами, регионами) и хотим быстро получить объекты в их пределах.

Давайте выгрузим здания в границах рассматриваемого района.


In [ ]:
tags = {'building': True}   

buildings = ox.features_from_place(area_name, tags)

buildings.explore(tiles='cartodbpositron',tooltip=None)

### 2.2 По границам полигона

Другой способ выгрузки данных — использовать заранее полученную геометрию. В этом случае мы передаём в запрос не название места, а конкретный полигон с границами.

Такой подход особенно удобен, когда границы сформированы вручную и не совпадают с административными районами.

Чтобы не загружать новые данные, воспользуемся границами района, которые мы получили в первом разделе. Результат будет идентичен, поскольку используется та же территория, но переданная в виде полигона.


In [ ]:
# district_polygon = admin_district.geometry.iloc[0]
# buildings = ox.features_from_polygon(district_polygon, tags)

# buildings.explore(tiles='cartodbpositron',tooltip=None)

### 2.3 По Bounding Box

Ещё один способ задания территории — использование **Bounding Box** (ограничивающего прямоугольника).

Bounding Box — это минимальный прямоугольник, который полностью охватывает выбранную геометрию. Он задаётся четырьмя координатами: северной, южной, восточной и западной границами.

Получим Bounding Box из границ района, который мы выгрузили в первом разделе, и выполним экспорт данных в его пределах.


In [ ]:

# выгрузим здания в пределах bounding box
tags = {"building": True}
buildings_bbox = ox.features_from_bbox(admin_district.total_bounds, tags)

buildings_bbox.head()


In [ ]:
buildings_bbox.explore(tiles='cartodbpositron',tooltip=None)

При использовании Bounding Box мы получили больше зданий, поскольку прямоугольник охватывает не только территорию района, но и прилегающие участки за его границами.


## 3. Изучение данных


После выгрузки объектов из OSM важно выполнить первичный обзор данных. Это позволяет понять их структуру, объём и качество перед дальнейшей обработкой и анализом.


### 3.1 Количество объектов

Сначала посмотрим, сколько объектов было загружено:


In [ ]:
len(buildings)

### 3.2 Типы геометрии

Проверим, какие геометрии присутствуют в данных:


In [ ]:
buildings.geom_type.unique()

В данных зданий могут встречаться `Polygon`, `MultiPolygon` и `Point`, потому что в OpenStreetMap одно и то же здание может быть представлено по-разному: как замкнутый контур, как составной объект из нескольких частей или как точка, если его границы не были прорисованы. Поэтому перед дальнейшим анализом важно определить, с какими типами геометрии мы будем работать, и при необходимости выполнить фильтрацию данных.


### 3.3 Атрибуты


Посмотрим, какие атрибуты содержит таблица и какие типы данных им соответствуют:


In [ ]:
dtypes_df = buildings.dtypes.reset_index()
dtypes_df.columns = ["column", "dtype"]

dtypes_df


## 4.Обработка


### 4.1 Фильтрация по типу геометрии

Как мы увидели ранее, в данных зданий могут встречаться разные типы геометрии (`Point`, `Polygon`, `MultiPolygon`). Для большинства пространственных операций (например, расчёта площади) нам подходят только полигоны.

Оставим только объекты с геометрией `Polygon` и `MultiPolygon`:


In [ ]:
buildings = buildings[
    buildings.geom_type.isin(["Polygon", "MultiPolygon"])
]

Проверим результат:


In [ ]:
buildings.geom_type.unique()

### 4.2 Проверка валидности геометрии


Перед дальнейшим анализом важно убедиться, что геометрии корректны.


In [ ]:
buildings.is_valid.value_counts()

Если присутствуют невалидные объекты, можно оставить только корректные:


In [ ]:
buildings = buildings[buildings.is_valid]

Или исправить некорректную геометрию:


In [ ]:
buildings["geometry"] = buildings.make_valid()

### 4.3 Удаление «лишних» полей

После выгрузки данных из OSM можно заметить, что количество полей в таблице довольно большое. Это связано с тем, что OpenStreetMap хранит множество тегов, и при экспорте мы получаем практически все атрибуты, которые были добавлены пользователями.

В результате таблица может содержать десятки полей, большинство из которых не нужны для текущей задачи. Всё зависит от того, как именно вы планируете использовать данные. Если, например, здания выступают как вспомогательный слой, нам могут быть важны только 1–2 ключевых атрибута.


In [ ]:
buildings = buildings[["id","building", "geometry"]]

Посмотрим на получившийся набор данных


In [ ]:
buildings.head()

## 5. Сохранение данных


Теперь можем сохранить полученный набор пространственных данных в любом поддерживаемом формате с помощью метода `to_file()` библиотеки GeoPandas.


In [ ]:
#buildings.to_file('../data/osm_build.geojson')

## 6. Итог


В этом разделе мы рассмотрели структуру данных OpenStreetMap и способы их выгрузки с помощью OSMnx.

Мы задали территорию исследования, сформировали запрос по тегам, и выполнили первичную обработку набора.
